# Lab 19: Tomografía de Estados Cuánticos

La **tomografía de estados cuánticos** (QST) es el proceso experimental de reconstruir la matriz densidad $\rho$ de un sistema cuántico a partir de medidas repetidas.

Para un sistema de $n$ qubits, $\rho$ tiene $4^n - 1$ parámetros reales independientes. La tomografía requiere medir en $4^n$ configuraciones de Pauli distintas.

Estudiaremos:
1. Tomografía de 1 qubit (3 parámetros: $\langle X\rangle, \langle Y\rangle, \langle Z\rangle$)
2. Tomografía de 2 qubits (15 parámetros)
3. Tomografía por máxima verosimilitud (MLE) para garantizar $\rho \geq 0$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product as iproduct
from scipy.optimize import minimize

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, DensityMatrix, state_fidelity, partial_trace
from qiskit.primitives import StatevectorSampler

np.random.seed(0)

# Matrices de Pauli
I2 = np.eye(2, dtype=complex)
X  = np.array([[0,1],[1,0]], dtype=complex)
Y  = np.array([[0,-1j],[1j,0]], dtype=complex)
Z  = np.array([[1,0],[0,-1]], dtype=complex)
paulis_1q = [I2, X, Y, Z]
pauli_labels = ['I', 'X', 'Y', 'Z']

## 1. Tomografía de 1 Qubit

Para un qubit, la matriz densidad se expande en la base de Pauli:
$$\rho = \frac{1}{2}(I + r_x X + r_y Y + r_z Z)$$
donde $\mathbf{r} = (\langle X\rangle, \langle Y\rangle, \langle Z\rangle)$ es el **vector de Bloch**.

In [ ]:
def tomography_1q(psi, shots=4096):
    """Reconstruye ρ de 1 qubit midiendo en bases X, Y, Z.""""
    sampler = StatevectorSampler()
    results = {}
    for axis in ['X', 'Y', 'Z']:
        qc = QuantumCircuit(1, 1)
        qc.initialize(psi, 0)
        if axis == 'X': qc.h(0)
        elif axis == 'Y': qc.sdg(0); qc.h(0)
        qc.measure(0, 0)
        counts = sampler.run([qc], shots=shots).result()[0].data.c.get_counts()
        p0 = counts.get('0', 0) / shots
        results[axis] = 2*p0 - 1
    rx, ry, rz = results['X'], results['Y'], results['Z']
    rho = 0.5 * (I2 + rx*X + ry*Y + rz*Z)
    return rho, (rx, ry, rz)

# Test con varios estados
test_cases = {
    '|0⟩': np.array([1,0], dtype=complex),
    '|1⟩': np.array([0,1], dtype=complex),
    '|+⟩': np.array([1,1], dtype=complex)/np.sqrt(2),
    '|i⟩': np.array([1,1j], dtype=complex)/np.sqrt(2),
}

print(f"{'Estado':>5} | {'rx':>7} {'ry':>7} {'rz':>7} | {'|r|':>6} | {'Fidelidad':>10}")
print("-"*55)
for name, psi in test_cases.items():
    rho_ideal = np.outer(psi, psi.conj())
    rho_rec, (rx, ry, rz) = tomography_1q(psi)
    r = np.sqrt(rx**2 + ry**2 + rz**2)
    fid = float(state_fidelity(DensityMatrix(rho_ideal), DensityMatrix(rho_rec)))
    print(f"{name:>5} | {rx:>7.3f} {ry:>7.3f} {rz:>7.3f} | {r:>6.4f} | {fid:>10.6f}")

## 2. Tomografía de 2 Qubits

Para 2 qubits, la base de Pauli tiene $4^2 = 16$ elementos $\{II, IX, IY, IZ, XI, XX, \ldots, ZZ\}$.
La matriz densidad $4\times 4$ se reconstruye como:

$$\rho = \frac{1}{4}\sum_{j,k \in \{I,X,Y,Z\}} \langle\sigma_j \otimes \sigma_k\rangle\, \sigma_j \otimes \sigma_k$$

In [ ]:
def tomography_2q(psi, shots=8192):
    """Reconstruye ρ de 2 qubits midiendo en los 9 pares (A,B) con A,B ∈ {X,Y,Z}.""""
    sampler = StatevectorSampler()
    expectations = {}
    for p0, p1 in iproduct(['X','Y','Z'], repeat=2):
        qc = QuantumCircuit(2, 2)
        qc.initialize(psi, [0,1])
        for qi, axis in enumerate([p0, p1]):
            if axis == 'X': qc.h(qi)
            elif axis == 'Y': qc.sdg(qi); qc.h(qi)
        qc.measure([0,1],[0,1])
        counts = sampler.run([qc], shots=shots).result()[0].data.c.get_counts()
        exp_val = sum((-1)**(int(b[0])+int(b[1])) * c for b,c in counts.items()) / shots
        expectations[(p0, p1)] = exp_val

    # Reconstruir ρ: usar completitud de base Pauli
    rho = np.eye(4, dtype=complex) / 4
    pauli_map = {'I': I2, 'X': X, 'Y': Y, 'Z': Z}
    for (p0, p1), ev in expectations.items():
        P = np.kron(pauli_map[p0], pauli_map[p1])
        rho += ev * P / 4

    return rho, expectations

# Estado de Bell |Φ+⟩
qc_bell = QuantumCircuit(2)
qc_bell.h(0); qc_bell.cx(0, 1)
psi_bell = Statevector(qc_bell).data

rho_ideal = np.outer(psi_bell, psi_bell.conj())
rho_rec, exp_dict = tomography_2q(psi_bell)

fid = float(state_fidelity(DensityMatrix(rho_ideal), DensityMatrix(rho_rec)))
print(f"Estado de Bell |Φ+⟩:")
print(f"  Fidelidad: {fid:.6f}")
print(f"  Traza:     {np.trace(rho_rec).real:.6f}")
print(f"  Pureza:    {float(DensityMatrix(rho_rec).purity()):.6f}")
print(f"
Valores esperados seleccionados:")
for key in [('Z','Z'),('X','X'),('Z','I'),('I','Z')]:
    print(f"  ⟨{key[0]}⊗{key[1]}⟩ = {exp_dict[key]:.4f}")

## 3. Tomografía por Máxima Verosimilitud (MLE)

El método de inversión lineal puede producir matrices densidad con valores propios negativos (no físicas). La **MLE** encuentra la $\rho$ física más probable que explica los datos observados.

Parametrizamos $\rho = T^\dagger T / \text{Tr}(T^\dagger T)$ donde $T$ es triangular inferior compleja, garantizando $\rho \geq 0$ y $\text{Tr}(\rho) = 1$.

In [ ]:
def mle_1q(psi, shots=512):
    """MLE para 1 qubit con shot noise.""""
    sampler = StatevectorSampler()
    data = {}
    for axis in ['X', 'Y', 'Z']:
        qc = QuantumCircuit(1, 1)
        qc.initialize(psi, 0)
        if axis == 'X': qc.h(0)
        elif axis == 'Y': qc.sdg(0); qc.h(0)
        qc.measure(0, 0)
        counts = sampler.run([qc], shots=shots).result()[0].data.c.get_counts()
        data[axis] = (counts.get('0',0), counts.get('1',0))

    # Parametrización Cholesky: T = [[a, 0], [b+ic, d]]
    def t_matrix(params):
        a, b, c, d = params
        return np.array([[a, 0], [b+1j*c, d]])

    def neg_loglik(params):
        T = t_matrix(params)
        rho = T.conj().T @ T; rho /= np.trace(rho)
        ll = 0.0
        for axis, (n0, n1) in data.items():
            if axis == 'X': P0 = np.outer([1,1],[1,1])/2
            elif axis == 'Y': P0 = np.outer([1,-1j],[1,1j])/2
            else: P0 = np.array([[1,0],[0,0]], dtype=complex)
            p0 = max(np.trace(P0 @ rho).real, 1e-12)
            p1 = max(1-p0, 1e-12)
            if n0 > 0: ll -= n0 * np.log(p0)
            if n1 > 0: ll -= n1 * np.log(p1)
        return ll

    res = minimize(neg_loglik, [0.7, 0.1, 0.1, 0.7], method='Nelder-Mead',
                   options={'maxiter': 5000, 'xatol': 1e-8})
    T = t_matrix(res.x)
    rho_mle = T.conj().T @ T; rho_mle /= np.trace(rho_mle)
    return rho_mle

# Comparación: inversión lineal vs MLE con pocos shots
psi_test = np.array([1, 1j], dtype=complex)/np.sqrt(2)  # |i⟩
rho_ideal = np.outer(psi_test, psi_test.conj())

rho_lin, _ = tomography_1q(psi_test, shots=128)
rho_mle    = mle_1q(psi_test, shots=128)

evals_lin = np.linalg.eigvalsh(rho_lin)
evals_mle = np.linalg.eigvalsh(rho_mle)

print("Comparación tomografía con 128 shots:")
print(f"{'Método':>18} | {'Fidelidad':>10} | {'Mín. eigenvalor':>16} | {'Física':>6}")
for method, rho_r in [('Inv. lineal', rho_lin), ('MLE', rho_mle)]:
    fid = float(state_fidelity(DensityMatrix(rho_ideal), DensityMatrix(rho_r)))
    evals = np.linalg.eigvalsh(rho_r)
    phys = evals.min() >= -1e-9
    print(f"{method:>18} | {fid:>10.6f} | {evals.min():>16.6f} | {'✓' if phys else '✗':>6}")

## 4. Limitaciones: Shot Noise y Escalado

El número de shots necesarios para una fidelidad $F \geq 1-\epsilon$ escala como $O(1/\epsilon^2)$. Para sistemas de $n$ qubits, el número de configuraciones Pauli crece como $4^n$, lo que hace la tomografía completa inviable para $n > 10$-$15$.

Alternativas para sistemas grandes: tomografía matricial de producto (MPT), compressed sensing, shadow tomography.

In [ ]:
# Fidelidad vs número de shots para estado |+⟩
shots_vals = [32, 64, 128, 256, 512, 1024, 2048, 4096]
psi_p = np.array([1, 1], dtype=complex)/np.sqrt(2)
rho_p_ideal = np.outer(psi_p, psi_p.conj())
fidelities_shots = []
n_trials = 5

for sh in shots_vals:
    fids_trial = []
    for _ in range(n_trials):
        rho_r, _ = tomography_1q(psi_p, shots=sh)
        fids_trial.append(float(state_fidelity(DensityMatrix(rho_p_ideal), DensityMatrix(rho_r))))
    fidelities_shots.append(np.mean(fids_trial))

plt.figure(figsize=(7, 4))
plt.semilogx(shots_vals, fidelities_shots, 'o-', color='steelblue', lw=2)
plt.axhline(0.999, color='red', ls='--', lw=1.5, label='F=0.999')
plt.xlabel('Número de shots', fontsize=12)
plt.ylabel('Fidelidad media', fontsize=12)
plt.title('Fidelidad de Tomografía vs Shots (1 qubit, |+⟩)')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print("
Escalado de recursos:")
print(f"{'n qubits':>10} | {'Parámetros':>12} | {'Configs Pauli':>14}")
for n in [1, 2, 3, 4, 5]:
    print(f"{n:>10} | {4**n - 1:>12} | {3**n:>14}")